# American Put Option Pricing - Monte Carlo LSM
## S&P 500 European vs American Comparison

This notebook prices American put options using Monte Carlo Least Squares Method (LSM).
Run each cell in order to see the full analysis.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

print("=" * 90)
print("AMERICAN PUT OPTION PRICING - MONTE CARLO SIMULATION")
print("=" * 90)

## Step 1: Define Option Parameters

Set the parameters for S&P 500 American put option

In [ ]:
# Real S&P 500 parameters (as of April 2026)
S0 = 5415.23          # Current S&P 500 price
K = 5400              # Strike price
T = 90 / 365.0        # Time to expiration (90 days)
r = 0.045             # Risk-free rate (4.5%)
sigma = 0.1845        # Volatility (18.45%)
q = 0.015             # Dividend yield (1.5%)

# Monte Carlo parameters
N_PATHS = 10000       # Number of simulated paths
N_STEPS = 90          # Number of time steps (daily)
dt = T / N_STEPS      # Time step size

print("\nOPTION PARAMETERS")
print("-" * 90)
print(f"Stock Price (S):           ${S0:>10,.2f}")
print(f"Strike Price (K):          ${K:>10,.2f}")
print(f"Time to Expiration (T):    {T:>10.4f} years ({T*365:.0f} days)")
print(f"Risk-free Rate (r):        {r:>10.2%}")
print(f"Volatility (sigma):        {sigma:>10.2%}")
print(f"Dividend Yield (q):        {q:>10.2%}")
print(f"\nMONTE CARLO SETTINGS")
print(f"Number of Paths:           {N_PATHS:>10,}")
print(f"Time Steps:                {N_STEPS:>10}")

## Step 2: Black-Scholes European Put (Baseline)

Calculate the baseline European put price using the closed-form Black-Scholes formula

In [ ]:
def black_scholes_put(S, K, r, sigma, T, q=0):
    """Calculate European put price using Black-Scholes formula"""
    d1 = (np.log(S/K) + (r - q + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    put_price = K*np.exp(-r*T)*norm.cdf(-d2) - S*np.exp(-q*T)*norm.cdf(-d1)
    return put_price

european_put_price = black_scholes_put(S0, K, r, sigma, T, q)

print(f"\n\nEUROPEAN PUT (Black-Scholes)")
print("-" * 90)
print(f"Price: ${european_put_price:.4f}")
print(f"Note: This assumes you hold until expiration (no early exercise)")

## Step 3: Monte Carlo American Put Pricing (LSM Method)

Price American put using Least Squares Monte Carlo

In [ ]:
def monte_carlo_american_put_lsm(S0, K, r, sigma, T, q, N_paths, N_steps):
    """
    Price American put using Least Squares Monte Carlo (LSM) method
    """
    dt = T / N_steps

    # Step 1: Generate stock price paths using Geometric Brownian Motion
    print("\nGenerating stock price paths...")

    paths = np.zeros((N_paths, N_steps + 1))
    paths[:, 0] = S0

    for t in range(1, N_steps + 1):
        Z = np.random.standard_normal(N_paths)
        paths[:, t] = paths[:, t-1] * np.exp(
            (r - q - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
        )

    print(f"[OK] Generated {N_paths:,} paths with {N_steps} steps each")

    # Step 2: Initialize option values at maturity
    print("Working backward from maturity...")
    option_values = np.maximum(K - paths[:, N_steps], 0)

    # Step 3: Backward induction (LSM algorithm)
    for t in range(N_steps - 1, 0, -1):
        S_t = paths[:, t]
        intrinsic = np.maximum(K - S_t, 0)
        discount = np.exp(-r * dt)
        ITM = intrinsic > 0

        if np.sum(ITM) > 0:
            S_itm = S_t[ITM]
            continuation_itm = option_values[ITM] * discount

            # Polynomial regression
            coeffs = np.polyfit(S_itm, continuation_itm, 3)
            poly = np.poly1d(coeffs)

            continuation = np.zeros(N_paths)
            continuation[ITM] = poly(S_itm)

            exercise = intrinsic > continuation

            option_values[exercise] = intrinsic[exercise]
            option_values[~exercise] = continuation[~exercise]
        else:
            option_values = option_values * discount

    american_put_price = np.mean(option_values) * np.exp(-r * dt)
    std_error = np.std(option_values) / np.sqrt(N_paths)

    return american_put_price, std_error, option_values, paths

print("\nRunning Monte Carlo simulation...")
american_put_price, std_error, option_values, paths = monte_carlo_american_put_lsm(
    S0, K, r, sigma, T, q, N_PATHS, N_STEPS
)
print(f"[OK] Simulation complete!")

## Step 4: Results and Analysis

In [ ]:
print("\n\nRESULTS: AMERICAN vs EUROPEAN PUT")
print("=" * 90)

early_exercise_premium = american_put_price - european_put_price
premium_percent = (early_exercise_premium / european_put_price) * 100

print(f"\nEuropean Put (Black-Scholes):  ${european_put_price:>8.4f}")
print(f"American Put (Monte Carlo):    ${american_put_price:>8.4f} +/- ${std_error:.4f}")
print(f"\nEarly Exercise Premium:        ${early_exercise_premium:>8.4f}")
print(f"Premium %:                     {premium_percent:>8.1f}%")

print(f"\n95% Confidence Interval:")
ci_lower = american_put_price - 1.96 * std_error
ci_upper = american_put_price + 1.96 * std_error
print(f"  ${ci_lower:.4f} to ${ci_upper:.4f}")

## Step 5: Path Analysis

In [ ]:
print("\n\nPATH ANALYSIS")
print("=" * 90)

final_prices = paths[:, -1]
min_prices = np.min(paths, axis=1)
max_prices = np.max(paths, axis=1)

print(f"\nStock Price Statistics (across {N_PATHS:,} paths):")
print(f"  Min price ever reached:    ${min_prices.mean():>8.2f} +/- ${min_prices.std():.2f}")
print(f"  Max price ever reached:    ${max_prices.mean():>8.2f} +/- ${max_prices.std():.2f}")
print(f"  Final price (day 90):      ${final_prices.mean():>8.2f} +/- ${final_prices.std():.2f}")

payoffs = np.maximum(K - final_prices, 0)
print(f"\nPayoff Statistics:")
print(f"  % paths in the money:      {100*np.sum(payoffs > 0)/N_PATHS:>7.1f}%")
print(f"  Average payoff if ITM:     ${payoffs[payoffs > 0].mean():>8.2f}")
print(f"  Average payoff overall:    ${payoffs.mean():>8.2f}")

## Step 6: Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Monte Carlo American Put Analysis - S&P 500', fontsize=16, fontweight='bold')

# Plot 1: Sample paths
ax = axes[0, 0]
for i in range(min(100, N_PATHS)):
    ax.plot(paths[i, :], alpha=0.1, color='blue')
ax.axhline(y=K, color='red', linestyle='--', linewidth=2, label=f'Strike = ${K}')
ax.axhline(y=S0, color='green', linestyle='--', linewidth=2, label=f'Current = ${S0:.0f}')
ax.set_xlabel('Days to Expiration')
ax.set_ylabel('Stock Price ($)')
ax.set_title('Sample Paths (100 of 10,000)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Distribution of final prices
ax = axes[0, 1]
ax.hist(final_prices, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=K, color='red', linestyle='--', linewidth=2, label='Strike')
ax.axvline(x=S0, color='green', linestyle='--', linewidth=2, label='Current')
ax.set_xlabel('Final Stock Price ($)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Final Prices (Day 90)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Payoff distribution
ax = axes[1, 0]
ax.hist(payoffs, bins=50, edgecolor='black', alpha=0.7, color='orange')
ax.set_xlabel('Put Payoff ($)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Final Payoffs')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Comparison
ax = axes[1, 1]
methods = ['European\nPut', 'American\nPut', 'Early Exercise\nPremium']
prices = [european_put_price, american_put_price, early_exercise_premium]
colors = ['lightblue', 'lightgreen', 'lightyellow']
bars = ax.bar(methods, prices, color=colors, edgecolor='black', linewidth=2)
ax.set_ylabel('Price ($)')
ax.set_title('American vs European Put Comparison')
ax.grid(True, alpha=0.3, axis='y')

for bar, price in zip(bars, prices):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${price:.2f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('american_put_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("[OK] Visualization complete!")

## Step 7: Key Insights

In [ ]:
print("\n" + "=" * 90)
print("KEY INSIGHTS")
print("=" * 90)

insights = f"""
1. EARLY EXERCISE PREMIUM
   American puts are worth ${early_exercise_premium:.2f} MORE than European
   This is {premium_percent:.1f}% premium for early exercise optionality

2. WHY SO MUCH PREMIUM?
   At current spot (${S0}), put is slightly OTM
   But with {sigma:.1%} volatility, can easily go deep ITM
   When deep ITM, early exercise becomes valuable
   Can lock in profits rather than waiting to expiration

3. MONTE CARLO vs BLACK-SCHOLES
   Black-Scholes (European):   ${european_put_price:.4f}  <- Wrong for American!
   Monte Carlo (American):     ${american_put_price:.4f}  <- Correct
   Difference:                 ${early_exercise_premium:.4f}  <- Money left on table!

4. REAL-WORLD APPLICATION
   If you're pricing American puts using Black-Scholes:
   -> You're underpricing by {premium_percent:.1f}%
   -> Clients think they're getting a great deal
   -> You're losing {premium_percent:.1f}% profit on every trade!
"""

print(insights)
print("=" * 90)

## Next Steps: Try Different Parameters

Edit the parameters above (S0, K, sigma, etc.) and re-run all cells to see how prices change.

Or test with real market data from:
- Yahoo Finance (free): finance.yahoo.com/quote/SPY/options
- Tastytrade (free app): www.tastytrade.com
- TD Ameritrade (free): www.tdameritrade.com